In [1]:
import gc
import os
import time
import numpy as np
import pandas as pd
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import mlflow
import pynvml


/home/yedhu/workspace/venvs/quantization-venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_proc = psutil.Process(os.getpid())

In [3]:
# 1. Define the dataset
PROMPTS = [
    ("simple_qa_1",   "What is the capital of Japan?", 32),
    ("simple_qa_2",   "Who wrote the play 'Hamlet'?", 32),
    ("reasoning_1",   "If a train travels 60 km in 45 minutes, what is its average speed in km/h? Explain.", 128),
    ("reasoning_2",   "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much is the ball? Reason step by step.", 128),
    ("coding_1",      "Write a Python function that returns the n-th Fibonacci number using memoization.", 200),
    ("coding_2",      "Write a SQL query to find the second highest salary from an Employees table.", 128),
    ("summarization", "Summarize the following in one sentence: Quantization reduces the numerical precision of model weights and activations, lowering memory and bandwidth needs while usually keeping accuracy close to the full-precision baseline.", 64),
    ("long_context",  "Here are five facts: (1) The sky is blue. (2) Water boils at 100C at sea level. (3) Paris is in France. (4) Honey does not spoil. (5) Spiders have eight legs. Which fact is about cooking, and which is about geography?", 96),
    ("math_1",        "Compute 17 * 23 and show the multiplication steps.", 96),
    ("math_2",        "What is the derivative of f(x) = 3x^2 + 2x - 5 with respect to x?", 64),
]

dataset = pd.DataFrame(PROMPTS, columns=["name", "prompt", "expected_len"])

In [4]:
MODEL_FP16 = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_AWQ  = "./models/llama-3.2-3b-instruct-awq-custom"
MODEL_GPTQ = "./models/llama-3.2-3b-instruct-gptq-custom"

# Metrics

In [5]:
def gpu_allocated_gb(device=DEVICE): return torch.cuda.memory_allocated() / 1024**3 if device == "cuda" else 0.0
def gpu_reserved_gb(device=DEVICE): return torch.cuda.memory_reserved() / 1024**3 if device == "cuda" else 0.0
def gpu_peak_reserved_gb(device=DEVICE): return torch.cuda.max_memory_reserved() / 1024**3 if device == "cuda" else 0.0
def gpu_peak_gb(device=DEVICE): return torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0.0

def cpu_resident_gb(): return _proc.memory_info().rss / 1024**3
def cpu_virtual_gb(): return _proc.memory_info().vms / 1024**3

In [6]:
def get_gpu_absolute_memory_usage():
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # Assuming single GPU
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return mem_info.used / 1024**3  # Convert bytes to GB

In [7]:
def get_gpu_blindspot_gb(device_index=0):
    if not torch.cuda.is_available():
        return 0.0
        
    # 1. Query the NVIDIA driver for the absolute total memory used 
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    total_system_used_gb = info.used / (1024**3)
    pynvml.nvmlShutdown()
    
    # 2. Query PyTorch for the total memory it is managing in its caching allocator
    pytorch_reserved_gb = torch.cuda.memory_reserved(device_index) / (1024**3)
    
    # 3. The difference is the untracked overhead
    blindspot_gb = total_system_used_gb - pytorch_reserved_gb
    
    return blindspot_gb

## Local Logging

In [8]:
metric_df = pd.DataFrame(
    columns=["LLAMA_3_2_3B_FP16", "LLAMA_3_2_3B_AWQ_CUSTOM","LLAMA_3_2_3B_GPTQ_CUSTOM"],
    index=pd.MultiIndex.from_tuples([], names=["metrics", "stage"])
)

In [9]:
def add_metric(df, metric, stage, model, value):
    # Create row if metric doesn't exist
    key = (metric, stage)
    if key not in df.index:
        df.loc[key, :] = [None] * len(df.columns)

    # Update the appropriate model column
    df.loc[key, model] = value
    return df

## MLFlow Logging

In [10]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("AWQ Quantization Benchmarking")
print(mlflow.get_tracking_uri())

http://localhost:5000


In [11]:
def mlflow_start_run(name: str):
    if mlflow.active_run():
        mlflow.end_run()
    return mlflow.start_run(run_name=name)

# Utils

In [12]:
def gpu_reset_peak(device=DEVICE): torch.cuda.reset_peak_memory_stats() if device == "cuda" else None

In [13]:
def clear_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# Timers

In [14]:
class BenchmarkTimer:
    def __init__(self, sync=True):
        self.sync = sync
        self.elapsed = None
    def __enter__(self):
        if self.sync and torch.cuda.is_available(): torch.cuda.synchronize()
        self._start = time.perf_counter()
        return self
    def __exit__(self, *exc):
        if self.sync and torch.cuda.is_available(): torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self._start
        return False

# Inference

## Custom TTFT Streamer

In [15]:
import time

class TTFTStreamer:
    """A minimal streamer designed solely to capture Time to First Token."""
    def __init__(self):
        self.start_time = None
        self.ttft = None
        self._call_count = 0

    def put(self, value):
        """Called by .generate() to push new tokens."""
        self._call_count += 1
        
        # Call 1: The input prompt
        # Call 2: The first newly generated token
        if self._call_count == 2 and self.start_time is not None:
            self.ttft = time.perf_counter() - self.start_time

    def end(self):
        """Called by .generate() to signal the end of generation."""
        pass

## Inference Definition

In [16]:
def run_inference(model_id, df, model_label):

    mlflow_start_run(f"Benchmarking {model_label}")

    print(f"\n==================================================")
    print(f"=== BENCHMARKING MODEL: {model_label} ===")
    print(f"==================================================")

    # --- CHECKPOINT 1: BASELINE (BEFORE LOADING) ---
    clear_cache()
    gpu_reset_peak()

    base_gpu = gpu_allocated_gb()
    base_cpu = cpu_resident_gb()
    print(f"[Baseline] GPU Allocated: {base_gpu:.3f} GB | CPU Resident: {base_cpu:.3f} GB")

    add_metric(metric_df, "gpu_allocated", "baseline", model_label, base_gpu)
    add_metric(metric_df, "cpu_resident", "baseline", model_label, base_cpu)
    mlflow.log_metric("baseline.gpu_allocated_gb", base_gpu)
    mlflow.log_metric("baseline.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("baseline.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("baseline.cpu_resident_gb", base_cpu)
    mlflow.log_metric("baseline.cpu_virtual_gb", cpu_virtual_gb())
    mlflow.log_metric("baseline.gpu_absolute_memory_usage_gb", get_gpu_absolute_memory_usage())
    mlflow.log_metric("baseline.gpu_blindspot_gb", get_gpu_blindspot_gb())

    print(f"\n=== Loading Model: {model_label} ({model_id}) ===")
    
    with BenchmarkTimer() as load_timer:
    # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        # Llama 3 models require setting the pad token if it's not set
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            
        # Load model (Transformers natively supports AWQ if autoawq is installed)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )


    # --- CHECKPOINT 2: POST-LOAD METRICS ---
    post_load_gpu = gpu_allocated_gb()
    reported_model_size_gb = (post_load_gpu - base_gpu) # Calculated delta via VRAM allocation
    
    print(f"[Loaded] Model Load Time: {load_timer.elapsed:.2f} seconds")
    print(f"[Loaded] Measured Model Weight Size: {reported_model_size_gb:.3f} GB")
    print(f"[Loaded] Current GPU Allocated: {post_load_gpu:.3f} GB | GPU Reserved: {gpu_reserved_gb():.3f} GB")

    add_metric(metric_df, "load_time", "post_load", model_label, load_timer.elapsed)
    add_metric(metric_df, "gpu_allocated", "post_load", model_label, post_load_gpu)
    add_metric(metric_df, "model_size_gb", "post_load", model_label, reported_model_size_gb)

    mlflow.log_metric("post_load.load_time_seconds", load_timer.elapsed)
    mlflow.log_metric("post_load.gpu_allocated_gb", post_load_gpu)
    mlflow.log_metric("post_load.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_load.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("post_load.reported_model_size_gb", reported_model_size_gb)
    mlflow.log_metric("post_load.gpu_absolute_memory_usage_gb", get_gpu_absolute_memory_usage())
    mlflow.log_metric("post_load.gpu_blindspot_gb", get_gpu_blindspot_gb())

    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size_gb = (param_size + buffer_size) / (1024 ** 3)
    
    
    print(f"-> Model Size in Memory: {total_size_gb:.2f} GB")

    add_metric(metric_df, "model_size_memory_gb", "post_load", model_label, total_size_gb)
    mlflow.log_metric("post_load.model_size_memory_gb", total_size_gb)

    print("\n[Warmup] Initializing CUDA context and JIT kernels...")
    
    # A tiny throwaway prompt
    warmup_messages = [{"role": "user", "content": "Hello."}]
    warmup_prompt = tokenizer.apply_chat_template(
        warmup_messages, tokenize=False, add_generation_prompt=True
    )
    warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(model.device)
    
    # Running fast generation to warm up the GPU and JIT kernels
    with torch.no_grad():
        _ = model.generate(
            **warmup_inputs,
            max_new_tokens=5, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Synchronize to ensure the GPU has completely finished the warmup task
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("[Warmup] Complete. GPU environment is ready.")
    # ---------------------------------------------------------

    # CHECKPOINT : POST WARMUP(BEFORE LOOP) ---
    clear_cache()
    gpu_reset_peak()

    mlflow.log_metric("warmup.gpu_allocated_gb", gpu_allocated_gb() )
    mlflow.log_metric("warmup.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("warmup.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("warmup.gpu_absolute_memory_usage_gb", get_gpu_absolute_memory_usage())
    mlflow.log_metric("warmup.gpu_blindspot_gb", get_gpu_blindspot_gb())

    results = []
    latencies = []
    ttft_list = []
    tokens_per_sec_list = []
    itl_list = []
    
    print(f"Running inference on {len(df)} samples...")
    for idx, row in df.iterrows():
        # Format the prompt using Llama 3's chat template for optimal Instruct performance
        messages = [{"role": "user", "content": row["prompt"]}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        
        # Instantiate the custom streamer for this specific prompt
        streamer = TTFTStreamer()
        # Generate text
        with BenchmarkTimer() as inf_timer:
            with torch.no_grad():
                # outputs = model.generate(
                #     **inputs,
                #     max_new_tokens=int(row["expected_len"]),
                #     do_sample=False, # Deterministic greedy decoding for benchmarking
                #     pad_token_id=tokenizer.eos_token_id
                # )
                # Start the TTFT clock exactly when generation starts
                streamer.start_time = time.perf_counter() 
                
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=int(row["expected_len"]),
                    min_new_tokens=int(row["expected_len"]), # Ensure we generate the expected number of tokens for benchmarking
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    streamer=streamer  
                )
        
        # Decode only the newly generated tokens
        generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]
        num_generated_tokens = len(generated_tokens)

        # If the model fails to generate anything, gracefully handle the NoneType
        ttft_seconds = streamer.ttft if streamer.ttft is not None else 0.0
        ttft_list.append(ttft_seconds)  # Store TTFT for this specific prompt
        # Calculate tokens per second (excluding the TTFT wait time for accuracy)
        generation_time = inf_timer.elapsed - ttft_seconds
        tokens_per_sec = (num_generated_tokens - 1) / generation_time if generation_time > 0 else 0
        itl_seconds = generation_time / (num_generated_tokens - 1)
        # print(f"Inter-Token Latency: {itl_seconds * 1000:.2f} ms")
        # tokens_per_sec = num_generated_tokens / inf_timer.elapsed if inf_timer.elapsed > 0 else 0
        tokens_per_sec_list.append(tokens_per_sec)

        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(response)
        latencies.append(inf_timer.elapsed)
        itl_list.append(itl_seconds)
        
        print(f"[{row['name']}] Done.")

        print(f"  -> [{row['name']}] Generated {num_generated_tokens} tokens in {inf_timer.elapsed:.3f}s ({tokens_per_sec:.2f} tok/s)")

        
 

        
    # --- CHECKPOINT 4: POST-INFERENCE / PEAK ANALYSIS ---
    peak_gpu = gpu_peak_gb()
    print(f"\n[Post-Inference Summary]")
    print(f"[*] Max Peak GPU Memory During Run: {peak_gpu:.3f} GB")
    print(f"[*] Total Execution VRAM Overhead (Peak - Baseline): {peak_gpu - base_gpu:.3f} GB")
    print(f"[*] Average Generation Latency: {sum(latencies)/len(latencies):.3f} seconds")

    add_metric(metric_df, "average_inference_latency", "post_inference", model_label, np.mean(latencies))
    add_metric(metric_df, "average_tokens_per_sec", "post_inference", model_label, np.mean(tokens_per_sec_list))
    add_metric(metric_df, "peak_gpu_allocated", "post_inference", model_label, peak_gpu)

    # mlflow.log_metric("post_inference.warmup_ttft_seconds", ttft_list[0])  # Log the TTFT of the first prompt as a representative warmup metric
    # mlflow.log_metric("post_inference.warmup_inference_latency_seconds", latencies[0])  # Log the latency of the first prompt as a representative warmup metric
    # mlflow.log_metric("post_inference.warmup_tokens_per_sec", tokens_per_sec_list[0])  # Log the tokens per second of the first prompt as a representative warmup metric
    mlflow.log_metric("post_inference.ttft_average_seconds", np.mean(ttft_list[1:]))  # Exclude the first prompt from the average to avoid skewing
    mlflow.log_metric("post_inference.average_inference_latency_seconds", np.mean(latencies[1:]))  # Exclude the first prompt from the average to avoid skewing
    mlflow.log_metric("post_inference.average_tokens_per_sec", np.mean(tokens_per_sec_list[1:]))  # Exclude the first prompt from the average to avoid skewing
    mlflow.log_metric("post_inference.average_inter_token_latency_seconds", np.mean(itl_list[1:]))  # Exclude the first prompt from the average to avoid skewing
    mlflow.log_metric("post_inference.peak_gpu_allocated_gb", peak_gpu)
    mlflow.log_metric("post_inference.total_execution_vram_overhead_gb", peak_gpu - base_gpu)
    mlflow.log_metric("post_inference.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_inference.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("post_inference.gpu_absolute_memory_usage_gb", get_gpu_absolute_memory_usage())
    mlflow.log_metric("post_inference.gpu_blindspot_gb", get_gpu_blindspot_gb())

    # Clean up GPU memory before the next model loads
    del model
    del tokenizer
    # torch.cuda.empty_cache()
    clear_cache()
    
    final_gpu = gpu_allocated_gb()
    print(f"[Cleanup] Post-Cleanup GPU Allocated: {final_gpu:.3f} GB (Should match or be close to Baseline)")
    add_metric(metric_df, "gpu_allocated", "post_cleanup", model_label, final_gpu)
    mlflow.log_metric("post_cleanup.gpu_allocated_gb", final_gpu)
    mlflow.log_metric("post_cleanup.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_cleanup.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("post_cleanup.gpu_absolute_memory_usage_gb", get_gpu_absolute_memory_usage())
    mlflow.log_metric("post_cleanup.gpu_blindspot_gb", get_gpu_blindspot_gb())

    print(f"\n=== Finished Benchmarking Model: {model_label} ===\n")
    print(f"==================================================\n")
    print(f"=== METRICS DATAFRAME ===")
    print(metric_df)

    return results

## FP16 Baseline Run

In [18]:
# 3. Run evaluation for both models
dataset["response_fp16"] = run_inference(MODEL_FP16, dataset, "LLAMA_3_2_3B_FP16")

🏃 View run Benchmarking LLAMA_3_2_3B_FP16 at: http://localhost:5000/#/experiments/6/runs/a9b75bcf77e34365a941587b7b0c9bf6
🧪 View experiment at: http://localhost:5000/#/experiments/6

=== BENCHMARKING MODEL: LLAMA_3_2_3B_FP16 ===
[Baseline] GPU Allocated: 0.000 GB | CPU Resident: 0.957 GB

=== Loading Model: LLAMA_3_2_3B_FP16 (meta-llama/Llama-3.2-3B-Instruct) ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[Loaded] Model Load Time: 66.96 seconds
[Loaded] Measured Model Weight Size: 5.984 GB
[Loaded] Current GPU Allocated: 5.984 GB | GPU Reserved: 6.004 GB
-> Model Size in Memory: 5.98 GB

[Warmup] Initializing CUDA context and JIT kernels...
[Warmup] Complete. GPU environment is ready.
Running inference on 10 samples...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[simple_qa_1] Done.
  -> [simple_qa_1] Generated 32 tokens in 0.370s (90.12 tok/s)
[simple_qa_2] Done.
  -> [simple_qa_2] Generated 32 tokens in 0.357s (90.32 tok/s)
[reasoning_1] Done.
  -> [reasoning_1] Generated 128 tokens in 1.437s (89.26 tok/s)
[reasoning_2] Done.
  -> [reasoning_2] Generated 128 tokens in 1.428s (89.91 tok/s)
[coding_1] Done.
  -> [coding_1] Generated 200 tokens in 2.227s (89.92 tok/s)
[coding_2] Done.
  -> [coding_2] Generated 128 tokens in 1.425s (89.93 tok/s)
[summarization] Done.
  -> [summarization] Generated 64 tokens in 0.735s (87.46 tok/s)
[long_context] Done.
  -> [long_context] Generated 96 tokens in 1.100s (87.63 tok/s)
[math_1] Done.
  -> [math_1] Generated 96 tokens in 1.080s (89.08 tok/s)
[math_2] Done.
  -> [math_2] Generated 64 tokens in 0.720s (89.22 tok/s)

[Post-Inference Summary]
[*] Max Peak GPU Memory During Run: 6.023 GB
[*] Total Execution VRAM Overhead (Peak - Baseline): 6.023 GB
[*] Average Generation Latency: 1.088 seconds
[Cleanup] Pos

## AWQ Run

In [18]:
dataset["response_awq"] = run_inference(MODEL_AWQ, dataset, "LLAMA_3_2_3B_AWQ_CUSTOM")

🏃 View run Benchmarking LLAMA_3_2_3B_FP16 at: http://localhost:5000/#/experiments/6/runs/e9de9f3bbd2041bcb96e16b9e06762c4
🧪 View experiment at: http://localhost:5000/#/experiments/6

=== BENCHMARKING MODEL: LLAMA_3_2_3B_AWQ_CUSTOM ===
[Baseline] GPU Allocated: 0.008 GB | CPU Resident: 2.058 GB

=== Loading Model: LLAMA_3_2_3B_AWQ_CUSTOM (./models/llama-3.2-3b-instruct-awq-custom) ===



WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0+8583076
Transformers : 5.12.1
Torch        : 2.11.0+cu130
Triton       : 3.6.0


INFO  Kernel: Auto-selection: adding candidate `AwqMarlinLinear`               


INFO  Kernel: selected -> `AwqMarlinLinear`.                                   


Loading weights:   0%|          | 0/646 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 5867 objects in 0.236s                            


[Loaded] Model Load Time: 2.45 seconds
[Loaded] Measured Model Weight Size: 2.098 GB
[Loaded] Current GPU Allocated: 2.106 GB | GPU Reserved: 2.158 GB
-> Model Size in Memory: 2.10 GB
Running inference on 10 samples...
[simple_qa_1] Done.
  -> [simple_qa_1] Generated 8 tokens in 0.097s (100.56 tok/s)
[simple_qa_2] Done.
  -> [simple_qa_2] Generated 13 tokens in 0.128s (102.57 tok/s)
[reasoning_1] Done.
  -> [reasoning_1] Generated 107 tokens in 1.042s (102.94 tok/s)
[reasoning_2] Done.
  -> [reasoning_2] Generated 128 tokens in 1.244s (103.26 tok/s)
[coding_1] Done.
  -> [coding_1] Generated 200 tokens in 1.951s (102.62 tok/s)
[coding_2] Done.
  -> [coding_2] Generated 128 tokens in 1.245s (103.05 tok/s)
[summarization] Done.
  -> [summarization] Generated 30 tokens in 0.295s (103.13 tok/s)
[long_context] Done.
  -> [long_context] Generated 96 tokens in 0.927s (104.41 tok/s)
[math_1] Done.
  -> [math_1] Generated 85 tokens in 0.808s (105.42 tok/s)
[math_2] Done.
  -> [math_2] Generated

## GPTQ Run

In [17]:
dataset["response_gptq"] = run_inference(MODEL_GPTQ, dataset, "LLAMA_3_2_3B_GPTQ_CUSTOM")


=== BENCHMARKING MODEL: LLAMA_3_2_3B_GPTQ_CUSTOM ===
[Baseline] GPU Allocated: 0.000 GB | CPU Resident: 0.938 GB

=== Loading Model: LLAMA_3_2_3B_GPTQ_CUSTOM (./models/llama-3.2-3b-instruct-gptq-custom) ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


...

Loading weights:   0%|          | 0/842 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 140 objects in 0.231s                             


[Loaded] Model Load Time: 4.29 seconds
[Loaded] Measured Model Weight Size: 2.088 GB
[Loaded] Current GPU Allocated: 2.088 GB | GPU Reserved: 2.156 GB
-> Model Size in Memory: 2.09 GB
Running inference on 10 samples...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[simple_qa_1] Done.
  -> [simple_qa_1] Generated 8 tokens in 0.293s (89.00 tok/s)
[simple_qa_2] Done.
  -> [simple_qa_2] Generated 13 tokens in 0.125s (105.53 tok/s)
[reasoning_1] Done.
  -> [reasoning_1] Generated 83 tokens in 0.781s (106.89 tok/s)
[reasoning_2] Done.
  -> [reasoning_2] Generated 128 tokens in 1.188s (108.35 tok/s)
[coding_1] Done.
  -> [coding_1] Generated 200 tokens in 1.852s (108.21 tok/s)
[coding_2] Done.
  -> [coding_2] Generated 128 tokens in 1.178s (108.99 tok/s)
[summarization] Done.
  -> [summarization] Generated 29 tokens in 0.270s (110.07 tok/s)
[long_context] Done.
  -> [long_context] Generated 96 tokens in 0.892s (108.87 tok/s)
[math_1] Done.
  -> [math_1] Generated 85 tokens in 0.788s (108.22 tok/s)
[math_2] Done.
  -> [math_2] Generated 64 tokens in 0.592s (108.84 tok/s)

[Post-Inference Summary]
[*] Max Peak GPU Memory During Run: 2.124 GB
[*] Total Execution VRAM Overhead (Peak - Baseline): 2.124 GB
[*] Average Generation Latency: 0.796 seconds
[Clean

In [7]:
# 4. Display or save results
print("\n=== Sample Comparison ===")
for idx, row in dataset.head(10).iterrows():
    print(f"\nPrompt Name: {row['name']}")
    print(f"Prompt: {row['prompt']}")
    print(f"-> FP16 Response: {row['response_fp16']}")
    print(f"-> AWQ Response: {row['response_awq']}")


=== Sample Comparison ===

Prompt Name: simple_qa_1
Prompt: What is the capital of Japan?
-> FP16 Response: The capital of Japan is Tokyo.
-> AWQ Response: The capital of Japan is Tokyo.

Prompt Name: simple_qa_2
Prompt: Who wrote the play 'Hamlet'?
-> FP16 Response: The play "Hamlet" was written by William Shakespeare.
-> AWQ Response: The play "Hamlet" was written by William Shakespeare.

Prompt Name: reasoning_1
Prompt: If a train travels 60 km in 45 minutes, what is its average speed in km/h? Explain.
-> FP16 Response: To find the average speed of the train, we need to convert the time from minutes to hours.

Time in hours = 45 minutes / 60 = 0.75 hours

Now, we can use the formula:

Average Speed = Total Distance / Total Time
= 60 km / 0.75 hours
= 80 km/h

So, the average speed of the train is 80 km/h.
-> AWQ Response: To find the average speed of the train, we need to convert the time from minutes to hours.

Time = 45 minutes = 45/60 hours = 0.75 hours

Distance = 60 km

Averag